# Model Load and Evaluation Tool

In this notebook we load pre-trained models, using parameters saved to WandB during training, and run the the same evaluation functions as used immediately after the training process. This demonstrates that saved models can be reloaded to produce the same results as reported after training. This also serves as a template for manipulating the pre-trained models in downstream applications.

## Setup

In [1]:
from flax import nnx
import jax
import jax.numpy as jnp
import pandas as pd
import glob
from pathlib import Path
import json
import os
import sys
import shlex
from collections import defaultdict
sys.path.append("..")

from qnrl.algorithms.algorithm import Algorithm

import wandb
api = wandb.Api()

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


In [2]:
root_sweeps = Path("../../wandb-sweep-data-qnrl/")
root_sweeps.mkdir(parents=True, exist_ok=True)

In [3]:
project_id = "qnrl"

In [4]:
sweep_name_id = { # name: id
    # QnRL.
    'qnrl/FrozenLake-v1': '7yoeyqqr',
    'qnrl/CliffWalking-v1': 'nin93kq8',
    'qnrl/Acrobot-v1': '5ehazde3',
    'qnrl/CartPole-v1': 'qojis1fw',
    'qnrl/Breakout-v5': 'eigl7wk5',
    'qnrl/SpaceInvaders-v5': '8adr9d1k',

    # C51.
    'c51/FrozenLake-v1': 'jzduwzvg',
    'c51/CliffWalking-v1': '8xyt1eb5',
    'c51/Acrobot-v1': 'n8imxz1v',
    'c51/CartPole-v1': 'arv708xs',
    'c51/Breakout-v5': 'dg4sodfc',
    'c51/SpaceInvaders-v5': 'wbmr06xp',
    
    # qC51.
    'qc51/FrozenLake-v1': 'n93n0nr1',
    'qc51/CliffWalking-v1': 'gicedw73',
    'qc51/Acrobot-v1': 'h0z9k7g5',
    'qc51/CartPole-v1': 'foiafm3w',
    'qc51/Breakout-v5': 'sya2ty4z',
    'qc51/SpaceInvaders-v5': 'vapn48ds',
}
best_runs_for_sweep = { # name: [id]
    # QnRL.
    "qnrl/FrozenLake-v1": ['xy1qeozs', 'nrg58af6', 'ohbnwer4', '1w6f17u8'],
    "qnrl/CliffWalking-v1": ['84ly3kb8', 'q14zmjpp', 'ujlm00gt', 'er4d6rnz'],
    "qnrl/Acrobot-v1": ['k9tdwgod', '7iae54v0', 'n7t9tfkv', 'se6p60c1'],
    "qnrl/CartPole-v1": ['0jash5pr', '8pi89rle', 'zds1hi8i', 'tqeh89te'],
    'qnrl/Breakout-v5': ['i8ijgd1m', 'syrhxro1', '4pc6bthv', '3pv6knud'],
    'qnrl/SpaceInvaders-v5': ['kiwbzgdv', '4kkrr3v5', '2artbn9x', 'nvmd7j1x'],
    
    # C51.
    "c51/FrozenLake-v1": ['907k571g', 'i8w40ma5', '8p7uzbpv', '8oqjux3k'],
    "c51/CliffWalking-v1": ['zko23iy1', 'd962vucv', 'azhg19ww', '0nn68l3j'],
    "c51/Acrobot-v1": ['zbzemhwh', 'hs1wv6fh', 'ec4ti73g', 'y18igax1'],
    "c51/CartPole-v1": ['ubxwaw3p', 'f20h31mw', 'gdixmx80', '9eihhl7r'],
    'c51/Breakout-v5': ['bxdtzg0b', 'cmn3go73', '62raq6de', '2vxp36ul'],
    'c51/SpaceInvaders-v5': ['wq1meahd', 'gbkustul', 'q75ssoqq', 'ekmn2epv'],
    
    # qC51.
    'qc51/FrozenLake-v1': ['3rtnigpu', 'umicozu1', 'yctwup6e', 'iwxfz5pm'],
    'qc51/CliffWalking-v1': ['1zq59brf', '8bu5s27o', 'gef2kaqk', 'k8ib05bx'],
    'qc51/Acrobot-v1': ['j9cjekb5', 'dr6e9vsy', '7c8lqsdf', 'f2zxnr8i'],
    'qc51/CartPole-v1': ['hexy916u', 'zx83s9lw', 'gk5ptvmf', 'i0s8x3ll'],
    'qc51/Breakout-v5': ['k4hf9d66', 'sq95jzsj', 'o0fl03ok', 'zjjotl7m'],
    'qc51/SpaceInvaders-v5': ['0p1x80d8', 'qw51d2ne', '0cu0ricr', 'az4d9xv3'],
}

In [5]:
import yaml
import ast
from qnrl.serialization import hash_serializable_object
from typing import Any

def get_run_config(config_file: Path):
    with open(str(config_file), 'r') as f:
        config = yaml.safe_load(f)
    return config

def get_run_metadata(metadata_file: Path):
    # Load metadata as JSON dictionary.
    with open(str(metadata_file), 'r') as f:
        metadata = json.load(f)
    return metadata

def get_run_args(metadata_file: Path) -> dict:
    metadata = get_run_metadata(metadata_file)
    l: list[tuple[str, str]] = [tokens.split('=', 1) for tokens in metadata['args'] if '--' in tokens and '=' in tokens]
    d = dict((key.replace('--', '', 1), ast.literal_eval(val)) for key, val in l)
    return d

def get_run_config_dict_from_metadata(metadata_file: Path) -> dict:
    """Compute run config hash from original run program command string."""
    arg_dict = get_run_args(metadata_file)
    return arg_dict['config']
    
    
    # metadata = get_run_metadata(metadata_file)
    
    # # Get config string from run program command arguments.
    # # Format is: --config="{...}"
    # cfg_str = metadata['args'][1].split("--config=")[1]
    # # Convert this string to dictionary via literal evaluation.
    # cfg_dict = ast.literal_eval(cfg_str)
    # return cfg_dict

In [6]:
from typing import Iterator

def eval(sweep_id: str, run_id: str, eval_kwargs_override: dict = {}) -> Iterator[dict]:
    
    dir_run = root_sweeps/f"sweep-{sweep_id}"/f"run-{run_id}"
    metadata_file = dir_run/"wandb-metadata.json"

    arg_dict = get_run_args(metadata_file)
    
    print(f"Evaluating: sweep={sweep_id}, run={run_id}, num_eval={arg_dict['num_eval']}")

    cfg = arg_dict['config']
    # cfg_hash, _ = hash_serializable_object(cfg) # Compute hash for config object.
    algo_config = Algorithm.from_dict(cfg)
    
    # Load parameters from model save.
    model_dirs = {Path(root).stem: Path(root) for root in glob.glob(str(dir_run/"models"/"*"))}
    algo_config.load(model_dirs)
    # nnx.display(nnx.state(algo_config.model))
    
    eval_reports = []
    eval_seed = arg_dict['eval_seed'] if arg_dict['eval_seed'] is not None else algo_config.seed
    key = jax.random.PRNGKey(seed=eval_seed)
    for eval_idx in range(arg_dict['num_eval']):
        
        # Generate a new random key.
        key, subkey = jax.random.split(key=key)
        
        # Evaluate the model using a different random seed.
        seed_eval = int(subkey[0])
        eval_kwargs = {}
        if 'eval_total_timesteps' in arg_dict and arg_dict['eval_total_timesteps'] is not None:
            eval_kwargs['total_timesteps'] = arg_dict['eval_total_timesteps']
        _, eval_report = algo_config.eval(seed=seed_eval, **{**eval_kwargs, **eval_kwargs_override})
        
        # Add an index counter for the evaluation report.
        eval_report['eval/index'] = eval_idx
        
        yield eval_report


def eval_via_env(env_key: str, eval_kwargs_override: dict = {}):
    eval_reports = defaultdict(lambda: defaultdict(list[dict])) # { sweep_id: { run_id: [report_0, report_1, ...] } }
    sweep_id = sweep_name_id[env_key]
    for run_id in best_runs_for_sweep[env_key]:
        eval_reports[sweep_id][run_id] = list(eval(sweep_id, run_id, eval_kwargs_override))
    return eval_reports

In [7]:
def get_metric_as_series_for_run(metric_key: str, run_id: str, eval_reports: list[dict]):
    """Extract metric key as Pandas series and change metric key to row ID."""
    return pd.DataFrame(eval_reports)[metric_key].rename(run_id)

def get_metric_as_dataframe_for_sweep(report_dict: dict, metric_key: str, sweep_id: str):
    l = []
    for run_id, eval_reports in report_dict[sweep_id].items():
        # l.append(pd.DataFrame(eval_reports)['eval/episode_reward'].to_list())
        l.append(pd.DataFrame(eval_reports)[metric_key].rename(run_id))
    df = pd.concat(l, axis=1, join='outer') # column keys are 'row_id'.
    return df

In [8]:
def display_report_results(report_dict: dict, metric_key: str):
    for sweep_id in report_dict.keys():
        df = get_metric_as_dataframe_for_sweep(report_dict, metric_key, sweep_id)
        print(f"[{sweep_id}] DataFrame:")
        print(df)
        colmeans = df.mean() # Preserve for downstream post-processing.
        print(f"[{sweep_id}] Column Means:")
        print(colmeans)
        colmeans = jnp.asarray(colmeans.tolist()) # Convert to numpy list.
        print(f"[{sweep_id}] Mean of Column Means: {colmeans.mean()}")
        print(f"[{sweep_id}] Std. of Column Means: {colmeans.std(ddof=0)}")

## Evaluation Results

In [9]:
# Epsilon to use for all evaluations.
eps = 0.0 # What we use for eval.
# eps = 0.05 # What Bellemare used for training.
# eps = 0.001 # What Bellemare used for eval.

# Metric key to use for report display.
metric_key_for_report_display = 'eval/episode_reward'

# Keep track of all reports for all evaluations.
all_report_dicts = {}

### CartPole

In [10]:
env_key = "c51/CartPole-v1"
all_report_dicts[env_key] = eval_via_env(env_key, eval_kwargs_override=dict(eps=eps))
display_report_results(all_report_dicts[env_key], metric_key_for_report_display)

Evaluating: sweep=arv708xs, run=ubxwaw3p, num_eval=10


  0%|          | 0/100000 [00:00<?, ?step/s]/Users/derieux/research/qnrl/.venv/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
  0%|          | 242/100000 [00:00<01:35, 1039.49step/s, eval/step=242, eval/episode_reward=243, eval/episode_length=243, eval/episode_time=0.232]


Evaluating: sweep=arv708xs, run=f20h31mw, num_eval=10


  0%|          | 459/100000 [00:00<01:31, 1085.10step/s, eval/step=459, eval/episode_reward=460, eval/episode_length=460, eval/episode_time=0.423]


Evaluating: sweep=arv708xs, run=gdixmx80, num_eval=10


  0%|          | 306/100000 [00:00<01:27, 1141.73step/s, eval/step=306, eval/episode_reward=307, eval/episode_length=307, eval/episode_time=0.268]


Evaluating: sweep=arv708xs, run=9eihhl7r, num_eval=10


  0%|          | 499/100000 [00:00<01:28, 1126.01step/s, eval/step=499, eval/episode_reward=500, eval/episode_length=500, eval/episode_time=0.443]


[arv708xs] DataFrame:
   ubxwaw3p  f20h31mw  gdixmx80  9eihhl7r
0     235.0     500.0     291.0     500.0
1     283.0     272.0     500.0     500.0
2     250.0     277.0     500.0     500.0
3     268.0     236.0     500.0     500.0
4     292.0     287.0     500.0     500.0
5     241.0     500.0     327.0     500.0
6     307.0     228.0     500.0     500.0
7     309.0     234.0     500.0     500.0
8     238.0     404.0     500.0     500.0
9     243.0     460.0     307.0     500.0
[arv708xs] Column Means:
ubxwaw3p    266.6
f20h31mw    339.8
gdixmx80    442.5
9eihhl7r    500.0
dtype: float64
[arv708xs] Mean of Column Means: 387.225
[arv708xs] Std. of Column Means: 90.24002368683199


In [11]:
env_key = "qc51/CartPole-v1"
all_report_dicts[env_key] = eval_via_env(env_key, eval_kwargs_override=dict(eps=eps))
display_report_results(all_report_dicts[env_key], metric_key_for_report_display)

Evaluating: sweep=foiafm3w, run=hexy916u, num_eval=10


  0%|          | 499/100000 [00:00<01:36, 1032.39step/s, eval/step=499, eval/episode_reward=500, eval/episode_length=500, eval/episode_time=0.483]


Evaluating: sweep=foiafm3w, run=zx83s9lw, num_eval=10


  0%|          | 163/100000 [00:00<01:52, 887.03step/s, eval/step=163, eval/episode_reward=164, eval/episode_length=164, eval/episode_time=0.183]


Evaluating: sweep=foiafm3w, run=gk5ptvmf, num_eval=10


  0%|          | 499/100000 [00:00<01:38, 1009.66step/s, eval/step=499, eval/episode_reward=500, eval/episode_length=500, eval/episode_time=0.494]


Evaluating: sweep=foiafm3w, run=i0s8x3ll, num_eval=10


  0%|          | 499/100000 [00:00<01:46, 932.18step/s, eval/step=499, eval/episode_reward=500, eval/episode_length=500, eval/episode_time=0.535] 


[foiafm3w] DataFrame:
   hexy916u  zx83s9lw  gk5ptvmf  i0s8x3ll
0     500.0     172.0     500.0     500.0
1     500.0     189.0     500.0     500.0
2     500.0     184.0     500.0     500.0
3     500.0     203.0     500.0     500.0
4     500.0     181.0     500.0     500.0
5     500.0     175.0     500.0     500.0
6     500.0     183.0     500.0     500.0
7     500.0     179.0     500.0     500.0
8     500.0     181.0     500.0     500.0
9     500.0     164.0     500.0     500.0
[foiafm3w] Column Means:
hexy916u    500.0
zx83s9lw    181.1
gk5ptvmf    500.0
i0s8x3ll    500.0
dtype: float64
[foiafm3w] Mean of Column Means: 420.275
[foiafm3w] Std. of Column Means: 138.08775063342875


In [12]:
env_key = "qnrl/CartPole-v1"
all_report_dicts[env_key] = eval_via_env(env_key, eval_kwargs_override=dict(eps=eps))
display_report_results(all_report_dicts[env_key], metric_key_for_report_display)

Evaluating: sweep=qojis1fw, run=0jash5pr, num_eval=10


  0%|          | 499/100000 [00:05<19:30, 85.02step/s, eval/step=499, eval/episode_reward=500, eval/episode_length=500, eval/episode_time=5.86]


Evaluating: sweep=qojis1fw, run=8pi89rle, num_eval=10


  0%|          | 499/100000 [00:06<20:16, 81.79step/s, eval/step=499, eval/episode_reward=500, eval/episode_length=500, eval/episode_time=6.09]


Evaluating: sweep=qojis1fw, run=zds1hi8i, num_eval=10


  0%|          | 454/100000 [00:05<19:45, 83.94step/s, eval/step=454, eval/episode_reward=455, eval/episode_length=455, eval/episode_time=5.4]


Evaluating: sweep=qojis1fw, run=tqeh89te, num_eval=10


  0%|          | 255/100000 [00:03<20:39, 80.49step/s, eval/step=255, eval/episode_reward=256, eval/episode_length=256, eval/episode_time=3.16]


[qojis1fw] DataFrame:
   0jash5pr  8pi89rle  zds1hi8i  tqeh89te
0     500.0     500.0     500.0     235.0
1     500.0     500.0     500.0     401.0
2     362.0     500.0     500.0     306.0
3     500.0     500.0     500.0     462.0
4     500.0     500.0     500.0     350.0
5     500.0     500.0     500.0     285.0
6     500.0     500.0     500.0     331.0
7     500.0     500.0     500.0     334.0
8     500.0     500.0     500.0     315.0
9     500.0     500.0     455.0     256.0
[qojis1fw] Column Means:
0jash5pr    486.2
8pi89rle    500.0
zds1hi8i    495.5
tqeh89te    327.5
dtype: float64
[qojis1fw] Mean of Column Means: 452.3
[qojis1fw] Std. of Column Means: 72.22496105917953


### FrozenLake

In [13]:
env_key = "c51/FrozenLake-v1"
all_report_dicts[env_key] = eval_via_env(env_key, eval_kwargs_override=dict(eps=eps))
display_report_results(all_report_dicts[env_key], metric_key_for_report_display)

Evaluating: sweep=jzduwzvg, run=907k571g, num_eval=10


  0%|          | 20/100000 [00:00<04:05, 407.82step/s, eval/step=20, eval/episode_reward=0.8, eval/episode_length=21, eval/episode_time=0.049]


Evaluating: sweep=jzduwzvg, run=i8w40ma5, num_eval=10


  0%|          | 21/100000 [00:00<03:25, 485.77step/s, eval/step=21, eval/episode_reward=0.79, eval/episode_length=22, eval/episode_time=0.043]


Evaluating: sweep=jzduwzvg, run=8p7uzbpv, num_eval=10


  0%|          | 21/100000 [00:00<03:10, 525.41step/s, eval/step=21, eval/episode_reward=0.79, eval/episode_length=22, eval/episode_time=0.0398]


Evaluating: sweep=jzduwzvg, run=8oqjux3k, num_eval=10


  0%|          | 21/100000 [00:00<03:25, 485.75step/s, eval/step=21, eval/episode_reward=0.79, eval/episode_length=22, eval/episode_time=0.0432]


[jzduwzvg] DataFrame:
   907k571g  i8w40ma5  8p7uzbpv  8oqjux3k
0      0.75      0.61     -0.58     -0.56
1      0.95      0.93      0.93     -0.24
2      0.71     -0.75     -1.00     -0.48
3     -0.57     -0.57     -0.57      0.67
4     -0.44      0.78      0.78     -0.41
5      0.64      0.63      0.63     -0.50
6     -0.48     -0.48     -0.48      0.70
7      0.72      0.71      0.71     -0.47
8      0.29     -0.46     -1.00      0.80
9      0.80      0.79      0.79      0.79
[jzduwzvg] Column Means:
907k571g    0.337
i8w40ma5    0.219
8p7uzbpv    0.021
8oqjux3k    0.030
dtype: float64
[jzduwzvg] Mean of Column Means: 0.15174999999999988
[jzduwzvg] Std. of Column Means: 0.13300258456135355


In [14]:
env_key = "qc51/FrozenLake-v1"
all_report_dicts[env_key] = eval_via_env(env_key, eval_kwargs_override=dict(eps=eps))
display_report_results(all_report_dicts[env_key], metric_key_for_report_display)

Evaluating: sweep=n93n0nr1, run=3rtnigpu, num_eval=10


  0%|          | 21/100000 [00:00<04:12, 396.53step/s, eval/step=21, eval/episode_reward=0.79, eval/episode_length=22, eval/episode_time=0.0527]


Evaluating: sweep=n93n0nr1, run=umicozu1, num_eval=10


  0%|          | 21/100000 [00:00<03:29, 477.00step/s, eval/step=21, eval/episode_reward=0.79, eval/episode_length=22, eval/episode_time=0.0439]


Evaluating: sweep=n93n0nr1, run=yctwup6e, num_eval=10


  0%|          | 21/100000 [00:00<03:42, 449.35step/s, eval/step=21, eval/episode_reward=0.79, eval/episode_length=22, eval/episode_time=0.0466]


Evaluating: sweep=n93n0nr1, run=iwxfz5pm, num_eval=10


  0%|          | 21/100000 [00:00<04:19, 384.83step/s, eval/step=21, eval/episode_reward=0.79, eval/episode_length=22, eval/episode_time=0.0545]


[n93n0nr1] DataFrame:
   3rtnigpu  umicozu1  yctwup6e  iwxfz5pm
0      0.61      0.61      0.61     -0.56
1      0.93      0.93      0.93     -0.24
2     -1.00     -0.75     -0.75     -0.48
3     -0.57     -0.57     -0.57      0.67
4      0.78      0.78      0.78     -0.41
5      0.63      0.63      0.63     -0.50
6     -0.48     -0.48     -0.48      0.70
7      0.71      0.71      0.71     -0.47
8      0.28     -0.46     -0.46      0.80
9      0.79      0.79      0.79      0.79
[n93n0nr1] Column Means:
3rtnigpu    0.268
umicozu1    0.219
yctwup6e    0.219
iwxfz5pm    0.030
dtype: float64
[n93n0nr1] Mean of Column Means: 0.1839999999999999
[n93n0nr1] Std. of Column Means: 0.09113451596404074


In [15]:
env_key = "qnrl/FrozenLake-v1"
all_report_dicts[env_key] = eval_via_env(env_key, eval_kwargs_override=dict(eps=eps))
display_report_results(all_report_dicts[env_key], metric_key_for_report_display)

Evaluating: sweep=7yoeyqqr, run=xy1qeozs, num_eval=10


  0%|          | 20/100000 [00:00<24:54, 66.88step/s, eval/step=20, eval/episode_reward=0.8, eval/episode_length=21, eval/episode_time=0.29]


Evaluating: sweep=7yoeyqqr, run=nrg58af6, num_eval=10


  0%|          | 52/100000 [00:00<21:44, 76.64step/s, eval/step=52, eval/episode_reward=0.48, eval/episode_length=53, eval/episode_time=0.669]


Evaluating: sweep=7yoeyqqr, run=ohbnwer4, num_eval=10


  0%|          | 20/100000 [00:00<26:15, 63.45step/s, eval/step=20, eval/episode_reward=0.8, eval/episode_length=21, eval/episode_time=0.303]


Evaluating: sweep=7yoeyqqr, run=1w6f17u8, num_eval=10


  0%|          | 20/100000 [00:00<24:05, 69.18step/s, eval/step=20, eval/episode_reward=0.8, eval/episode_length=21, eval/episode_time=0.28]


[7yoeyqqr] DataFrame:
   xy1qeozs  nrg58af6  ohbnwer4  1w6f17u8
0      0.75      0.76      0.75      0.75
1      0.95     -0.35      0.95      0.95
2      0.71      0.05      0.71      0.71
3      0.86     -0.57      0.86     -0.57
4     -0.44      0.68     -0.44     -0.44
5      0.64      0.69      0.64      0.64
6     -0.48     -0.48     -0.48     -0.48
7      0.72      0.45      0.72      0.72
8     -0.80     -0.45     -1.00     -0.46
9      0.80      0.48      0.80      0.80
[7yoeyqqr] Column Means:
xy1qeozs    0.371
nrg58af6    0.126
ohbnwer4    0.351
1w6f17u8    0.262
dtype: float64
[7yoeyqqr] Mean of Column Means: 0.27749999999999986
[7yoeyqqr] Std. of Column Means: 0.09661392239216872


### CliffWalking

In [16]:
env_key = "c51/CliffWalking-v1"
all_report_dicts[env_key] = eval_via_env(env_key, eval_kwargs_override=dict(eps=eps))
display_report_results(all_report_dicts[env_key], metric_key_for_report_display)

Evaluating: sweep=8xyt1eb5, run=zko23iy1, num_eval=10


  0%|          | 12/100000 [00:00<06:54, 241.35step/s, eval/step=12, eval/episode_reward=-13, eval/episode_length=13, eval/episode_time=0.0493]


Evaluating: sweep=8xyt1eb5, run=d962vucv, num_eval=10


  0%|          | 12/100000 [00:00<06:44, 247.35step/s, eval/step=12, eval/episode_reward=-13, eval/episode_length=13, eval/episode_time=0.0481]


Evaluating: sweep=8xyt1eb5, run=azhg19ww, num_eval=10


  0%|          | 12/100000 [00:00<05:39, 294.79step/s, eval/step=12, eval/episode_reward=-13, eval/episode_length=13, eval/episode_time=0.0402]


Evaluating: sweep=8xyt1eb5, run=0nn68l3j, num_eval=10


  0%|          | 12/100000 [00:00<06:35, 253.09step/s, eval/step=12, eval/episode_reward=-13, eval/episode_length=13, eval/episode_time=0.0457]


[8xyt1eb5] DataFrame:
   zko23iy1  d962vucv  azhg19ww  0nn68l3j
0     -13.0     -13.0     -13.0     -13.0
1     -13.0     -13.0     -13.0     -13.0
2     -13.0     -13.0     -13.0     -13.0
3     -13.0     -13.0     -13.0     -13.0
4     -13.0     -13.0     -13.0     -13.0
5     -13.0     -13.0     -13.0     -13.0
6     -13.0     -13.0     -13.0     -13.0
7     -13.0     -13.0     -13.0     -13.0
8     -13.0     -13.0     -13.0     -13.0
9     -13.0     -13.0     -13.0     -13.0
[8xyt1eb5] Column Means:
zko23iy1   -13.0
d962vucv   -13.0
azhg19ww   -13.0
0nn68l3j   -13.0
dtype: float64
[8xyt1eb5] Mean of Column Means: -13.0
[8xyt1eb5] Std. of Column Means: 0.0


In [17]:
env_key = "qc51/CliffWalking-v1"
all_report_dicts[env_key] = eval_via_env(env_key, eval_kwargs_override=dict(eps=eps))
display_report_results(all_report_dicts[env_key], metric_key_for_report_display)

Evaluating: sweep=gicedw73, run=1zq59brf, num_eval=10


  0%|          | 12/100000 [00:00<07:02, 236.81step/s, eval/step=12, eval/episode_reward=-13, eval/episode_length=13, eval/episode_time=0.0502]


Evaluating: sweep=gicedw73, run=8bu5s27o, num_eval=10


  0%|          | 12/100000 [00:00<06:54, 241.10step/s, eval/step=12, eval/episode_reward=-13, eval/episode_length=13, eval/episode_time=0.0493]


Evaluating: sweep=gicedw73, run=gef2kaqk, num_eval=10


  0%|          | 12/100000 [00:00<06:29, 256.69step/s, eval/step=12, eval/episode_reward=-13, eval/episode_length=13, eval/episode_time=0.0463]


Evaluating: sweep=gicedw73, run=k8ib05bx, num_eval=10


  0%|          | 12/100000 [00:00<06:48, 244.93step/s, eval/step=12, eval/episode_reward=-13, eval/episode_length=13, eval/episode_time=0.0486]


[gicedw73] DataFrame:
   1zq59brf  8bu5s27o  gef2kaqk  k8ib05bx
0     -13.0     -13.0     -13.0     -13.0
1     -13.0     -13.0     -13.0     -13.0
2     -13.0     -13.0     -13.0     -13.0
3     -13.0     -13.0     -13.0     -13.0
4     -13.0     -13.0     -13.0     -13.0
5     -13.0     -13.0     -13.0     -13.0
6     -13.0     -13.0     -13.0     -13.0
7     -13.0     -13.0     -13.0     -13.0
8     -13.0     -13.0     -13.0     -13.0
9     -13.0     -13.0     -13.0     -13.0
[gicedw73] Column Means:
1zq59brf   -13.0
8bu5s27o   -13.0
gef2kaqk   -13.0
k8ib05bx   -13.0
dtype: float64
[gicedw73] Mean of Column Means: -13.0
[gicedw73] Std. of Column Means: 0.0


In [18]:
env_key = "qnrl/CliffWalking-v1"
all_report_dicts[env_key] = eval_via_env(env_key, eval_kwargs_override=dict(eps=eps))
display_report_results(all_report_dicts[env_key], metric_key_for_report_display)

Evaluating: sweep=nin93kq8, run=84ly3kb8, num_eval=10


  0%|          | 12/100000 [00:00<27:19, 60.98step/s, eval/step=12, eval/episode_reward=-13, eval/episode_length=13, eval/episode_time=0.187]


Evaluating: sweep=nin93kq8, run=q14zmjpp, num_eval=10


  0%|          | 12/100000 [00:00<26:19, 63.31step/s, eval/step=12, eval/episode_reward=-13, eval/episode_length=13, eval/episode_time=0.18]


Evaluating: sweep=nin93kq8, run=ujlm00gt, num_eval=10


  0%|          | 12/100000 [00:00<28:12, 59.09step/s, eval/step=12, eval/episode_reward=-13, eval/episode_length=13, eval/episode_time=0.193]


Evaluating: sweep=nin93kq8, run=er4d6rnz, num_eval=10


  0%|          | 12/100000 [00:00<26:40, 62.48step/s, eval/step=12, eval/episode_reward=-13, eval/episode_length=13, eval/episode_time=0.182]


[nin93kq8] DataFrame:
   84ly3kb8  q14zmjpp  ujlm00gt  er4d6rnz
0     -13.0     -13.0     -13.0     -13.0
1     -13.0     -13.0     -13.0     -13.0
2     -13.0     -13.0     -13.0     -13.0
3     -13.0     -13.0     -13.0     -13.0
4     -13.0     -13.0     -13.0     -13.0
5     -13.0     -13.0     -13.0     -13.0
6     -13.0     -13.0     -13.0     -13.0
7     -13.0     -13.0     -13.0     -13.0
8     -13.0     -13.0     -13.0     -13.0
9     -13.0     -13.0     -13.0     -13.0
[nin93kq8] Column Means:
84ly3kb8   -13.0
q14zmjpp   -13.0
ujlm00gt   -13.0
er4d6rnz   -13.0
dtype: float64
[nin93kq8] Mean of Column Means: -13.0
[nin93kq8] Std. of Column Means: 0.0


### Acrobot

In [19]:
env_key = "c51/Acrobot-v1"
all_report_dicts[env_key] = eval_via_env(env_key, eval_kwargs_override=dict(eps=eps))
display_report_results(all_report_dicts[env_key], metric_key_for_report_display)

Evaluating: sweep=n8imxz1v, run=zbzemhwh, num_eval=10


  0%|          | 422/100000 [00:00<01:42, 974.29step/s, eval/step=422, eval/episode_reward=-422, eval/episode_length=423, eval/episode_time=0.433] 


Evaluating: sweep=n8imxz1v, run=hs1wv6fh, num_eval=10


  0%|          | 499/100000 [00:00<01:42, 975.28step/s, eval/step=499, eval/episode_reward=-500, eval/episode_length=500, eval/episode_time=0.511] 


Evaluating: sweep=n8imxz1v, run=ec4ti73g, num_eval=10


  0%|          | 179/100000 [00:00<02:00, 825.58step/s, eval/step=179, eval/episode_reward=-179, eval/episode_length=180, eval/episode_time=0.216]


Evaluating: sweep=n8imxz1v, run=y18igax1, num_eval=10


  0%|          | 218/100000 [00:00<01:50, 905.08step/s, eval/step=218, eval/episode_reward=-218, eval/episode_length=219, eval/episode_time=0.24]


[n8imxz1v] DataFrame:
   zbzemhwh  hs1wv6fh  ec4ti73g  y18igax1
0    -500.0    -500.0    -500.0    -122.0
1    -500.0    -500.0    -500.0    -113.0
2    -500.0    -500.0    -171.0    -102.0
3    -293.0    -500.0    -118.0     -99.0
4    -500.0    -500.0    -163.0    -113.0
5    -500.0    -500.0    -141.0    -168.0
6    -284.0    -500.0    -163.0    -100.0
7    -484.0    -500.0    -156.0    -148.0
8    -326.0    -500.0    -156.0     -97.0
9    -422.0    -500.0    -179.0    -218.0
[n8imxz1v] Column Means:
zbzemhwh   -430.9
hs1wv6fh   -500.0
ec4ti73g   -224.7
y18igax1   -128.0
dtype: float64
[n8imxz1v] Mean of Column Means: -320.9
[n8imxz1v] Std. of Column Means: 150.53376697605094


In [20]:
env_key = "qc51/Acrobot-v1"
all_report_dicts[env_key] = eval_via_env(env_key, eval_kwargs_override=dict(eps=eps))
display_report_results(all_report_dicts[env_key], metric_key_for_report_display)

Evaluating: sweep=h0z9k7g5, run=j9cjekb5, num_eval=10


  0%|          | 96/100000 [00:00<02:13, 750.09step/s, eval/step=96, eval/episode_reward=-96, eval/episode_length=97, eval/episode_time=0.127]


Evaluating: sweep=h0z9k7g5, run=dr6e9vsy, num_eval=10


  0%|          | 93/100000 [00:00<02:17, 724.80step/s, eval/step=93, eval/episode_reward=-93, eval/episode_length=94, eval/episode_time=0.128]


Evaluating: sweep=h0z9k7g5, run=7c8lqsdf, num_eval=10


  0%|          | 499/100000 [00:00<01:47, 928.24step/s, eval/step=499, eval/episode_reward=-500, eval/episode_length=500, eval/episode_time=0.537]


Evaluating: sweep=h0z9k7g5, run=f2zxnr8i, num_eval=10


  0%|          | 175/100000 [00:00<01:54, 875.24step/s, eval/step=175, eval/episode_reward=-175, eval/episode_length=176, eval/episode_time=0.199]


[h0z9k7g5] DataFrame:
   j9cjekb5  dr6e9vsy  7c8lqsdf  f2zxnr8i
0     -90.0     -92.0    -500.0    -228.0
1     -82.0     -93.0    -500.0    -188.0
2    -102.0     -87.0    -500.0    -500.0
3     -93.0     -94.0    -500.0    -430.0
4     -82.0     -93.0    -500.0    -356.0
5    -122.0     -87.0    -500.0    -212.0
6     -96.0     -84.0    -500.0    -500.0
7    -122.0     -78.0    -500.0    -500.0
8    -113.0    -115.0    -500.0    -263.0
9     -96.0     -93.0    -500.0    -175.0
[h0z9k7g5] Column Means:
j9cjekb5    -99.8
dr6e9vsy    -91.6
7c8lqsdf   -500.0
f2zxnr8i   -335.2
dtype: float64
[h0z9k7g5] Mean of Column Means: -256.65
[h0z9k7g5] Std. of Column Means: 171.19634195858274


In [21]:
env_key = "qnrl/Acrobot-v1"
all_report_dicts[env_key] = eval_via_env(env_key, eval_kwargs_override=dict(eps=eps))
display_report_results(all_report_dicts[env_key], metric_key_for_report_display)

Evaluating: sweep=5ehazde3, run=k9tdwgod, num_eval=10


  0%|          | 163/100000 [00:02<24:58, 66.60step/s, eval/step=163, eval/episode_reward=-163, eval/episode_length=164, eval/episode_time=2.44]


Evaluating: sweep=5ehazde3, run=7iae54v0, num_eval=10


  0%|          | 499/100000 [00:07<24:09, 68.64step/s, eval/step=499, eval/episode_reward=-500, eval/episode_length=500, eval/episode_time=7.26]


Evaluating: sweep=5ehazde3, run=n7t9tfkv, num_eval=10


  0%|          | 97/100000 [00:01<24:38, 67.57step/s, eval/step=97, eval/episode_reward=-97, eval/episode_length=98, eval/episode_time=1.43]


Evaluating: sweep=5ehazde3, run=se6p60c1, num_eval=10


  0%|          | 93/100000 [00:01<24:47, 67.16step/s, eval/step=93, eval/episode_reward=-93, eval/episode_length=94, eval/episode_time=1.37]


[5ehazde3] DataFrame:
   k9tdwgod  7iae54v0  n7t9tfkv  se6p60c1
0    -154.0    -500.0    -117.0    -112.0
1    -125.0    -136.0     -97.0     -69.0
2    -158.0    -500.0     -96.0     -70.0
3    -111.0    -500.0     -93.0     -99.0
4    -139.0    -500.0    -119.0    -112.0
5    -169.0    -159.0    -117.0     -96.0
6    -138.0    -500.0    -270.0     -70.0
7    -162.0    -500.0    -106.0     -69.0
8    -179.0    -500.0     -86.0     -76.0
9    -163.0    -500.0     -97.0     -93.0
[5ehazde3] Column Means:
k9tdwgod   -149.8
7iae54v0   -429.5
n7t9tfkv   -119.8
se6p60c1    -86.6
dtype: float64
[5ehazde3] Mean of Column Means: -196.42499999999998
[5ehazde3] Std. of Column Means: 136.4100138369614


### SpaceInvaders

In [22]:
env_key = "c51/SpaceInvaders-v5"
all_report_dicts[env_key] = eval_via_env(env_key, eval_kwargs_override=dict(eps=eps))
display_report_results(all_report_dicts[env_key], metric_key_for_report_display)

Evaluating: sweep=wbmr06xp, run=wq1meahd, num_eval=10


A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
 18%|█▊        | 182/1000 [00:00<00:00, 877.15step/s, eval/step=182, eval/episode_reward=155, eval/episode_length=183, eval/episode_time=0.208]


Evaluating: sweep=wbmr06xp, run=gbkustul, num_eval=10


 15%|█▌        | 151/1000 [00:00<00:00, 849.28step/s, eval/step=151, eval/episode_reward=70, eval/episode_length=152, eval/episode_time=0.178]


Evaluating: sweep=wbmr06xp, run=q75ssoqq, num_eval=10


 53%|█████▎    | 532/1000 [00:00<00:00, 840.06step/s, eval/step=532, eval/episode_reward=230, eval/episode_length=533, eval/episode_time=0.633]


Evaluating: sweep=wbmr06xp, run=ekmn2epv, num_eval=10


 21%|██        | 207/1000 [00:00<00:00, 868.21step/s, eval/step=207, eval/episode_reward=135, eval/episode_length=208, eval/episode_time=0.239]


[wbmr06xp] DataFrame:
   wq1meahd  gbkustul  q75ssoqq  ekmn2epv
0     210.0     150.0      45.0     125.0
1     245.0     175.0      55.0      70.0
2     245.0     175.0      55.0      70.0
3     155.0      70.0     230.0     135.0
4     210.0     150.0      45.0     125.0
5     155.0      70.0     230.0     135.0
6     155.0      70.0     230.0     135.0
7     155.0      70.0     230.0     135.0
8     155.0      70.0     230.0     135.0
9     155.0      70.0     230.0     135.0
[wbmr06xp] Column Means:
wq1meahd    184.0
gbkustul    107.0
q75ssoqq    158.0
ekmn2epv    120.0
dtype: float64
[wbmr06xp] Mean of Column Means: 142.25
[wbmr06xp] Std. of Column Means: 30.531745773866255


In [23]:
env_key = "qc51/SpaceInvaders-v5"
all_report_dicts[env_key] = eval_via_env(env_key, eval_kwargs_override=dict(eps=eps))
display_report_results(all_report_dicts[env_key], metric_key_for_report_display)

Evaluating: sweep=vapn48ds, run=0p1x80d8, num_eval=10


 22%|██▏       | 216/1000 [00:00<00:00, 866.32step/s, eval/step=216, eval/episode_reward=70, eval/episode_length=217, eval/episode_time=0.25]


Evaluating: sweep=vapn48ds, run=qw51d2ne, num_eval=10


 14%|█▍        | 143/1000 [00:00<00:01, 856.00step/s, eval/step=143, eval/episode_reward=80, eval/episode_length=144, eval/episode_time=0.167]


Evaluating: sweep=vapn48ds, run=0cu0ricr, num_eval=10


 16%|█▌        | 161/1000 [00:00<00:01, 769.56step/s, eval/step=161, eval/episode_reward=90, eval/episode_length=162, eval/episode_time=0.209]


Evaluating: sweep=vapn48ds, run=az4d9xv3, num_eval=10


 33%|███▎      | 326/1000 [00:00<00:00, 765.85step/s, eval/step=326, eval/episode_reward=315, eval/episode_length=327, eval/episode_time=0.426]


[vapn48ds] DataFrame:
   0p1x80d8  qw51d2ne  0cu0ricr  az4d9xv3
0      35.0     145.0      75.0      80.0
1      80.0      50.0     225.0      80.0
2      80.0      50.0     225.0      80.0
3      70.0      80.0      90.0     315.0
4      35.0     145.0      75.0      80.0
5      70.0      80.0      90.0     315.0
6      70.0      80.0      90.0     315.0
7      70.0      80.0      90.0     315.0
8      70.0      80.0      90.0     315.0
9      70.0      80.0      90.0     315.0
[vapn48ds] Column Means:
0p1x80d8     65.0
qw51d2ne     87.0
0cu0ricr    114.0
az4d9xv3    221.0
dtype: float64
[vapn48ds] Mean of Column Means: 121.75
[vapn48ds] Std. of Column Means: 59.87225985379206


In [24]:
env_key = "qnrl/SpaceInvaders-v5"
all_report_dicts[env_key] = eval_via_env(env_key, eval_kwargs_override=dict(eps=eps))
display_report_results(all_report_dicts[env_key], metric_key_for_report_display)

Evaluating: sweep=8adr9d1k, run=kiwbzgdv, num_eval=10


 19%|█▊        | 187/1000 [00:11<00:50, 16.05step/s, eval/step=187, eval/episode_reward=125, eval/episode_length=188, eval/episode_time=11.6]


Evaluating: sweep=8adr9d1k, run=4kkrr3v5, num_eval=10


 59%|█████▉    | 590/1000 [00:36<00:25, 16.07step/s, eval/step=590, eval/episode_reward=425, eval/episode_length=591, eval/episode_time=36.7]


Evaluating: sweep=8adr9d1k, run=2artbn9x, num_eval=10


 30%|███       | 301/1000 [00:18<00:44, 15.88step/s, eval/step=301, eval/episode_reward=120, eval/episode_length=302, eval/episode_time=18.9]


Evaluating: sweep=8adr9d1k, run=nvmd7j1x, num_eval=10


 19%|█▉        | 188/1000 [00:11<00:50, 16.19step/s, eval/step=188, eval/episode_reward=110, eval/episode_length=189, eval/episode_time=11.6]


[8adr9d1k] DataFrame:
   kiwbzgdv  4kkrr3v5  2artbn9x  nvmd7j1x
0      60.0      30.0     150.0      35.0
1     140.0     155.0      65.0     115.0
2     140.0     155.0      65.0     115.0
3     125.0     425.0     120.0     110.0
4      60.0      30.0     150.0      35.0
5     125.0     425.0     120.0     110.0
6     125.0     425.0     120.0     110.0
7     125.0     425.0     120.0     110.0
8     125.0     425.0     120.0     110.0
9     125.0     425.0     120.0     110.0
[8adr9d1k] Column Means:
kiwbzgdv    115.0
4kkrr3v5    292.0
2artbn9x    115.0
nvmd7j1x     96.0
dtype: float64
[8adr9d1k] Mean of Column Means: 154.5
[8adr9d1k] Std. of Column Means: 79.76371355447287


### Breakout

In [25]:
env_key = "c51/Breakout-v5"
all_report_dicts[env_key] = eval_via_env(env_key, eval_kwargs_override=dict(eps=eps))
display_report_results(all_report_dicts[env_key], metric_key_for_report_display)

Evaluating: sweep=dg4sodfc, run=bxdtzg0b, num_eval=10


 20%|█▉        | 198/1000 [00:00<00:00, 809.22step/s, eval/step=198, eval/episode_reward=6, eval/episode_length=199, eval/episode_time=0.245]


Evaluating: sweep=dg4sodfc, run=cmn3go73, num_eval=10


 15%|█▌        | 152/1000 [00:00<00:01, 776.33step/s, eval/step=152, eval/episode_reward=4, eval/episode_length=153, eval/episode_time=0.196]


Evaluating: sweep=dg4sodfc, run=62raq6de, num_eval=10


 27%|██▋       | 274/1000 [00:00<00:00, 757.51step/s, eval/step=274, eval/episode_reward=8, eval/episode_length=275, eval/episode_time=0.362]


Evaluating: sweep=dg4sodfc, run=2vxp36ul, num_eval=10


  5%|▌         | 50/1000 [00:00<00:01, 760.69step/s, eval/step=50, eval/episode_reward=1, eval/episode_length=51, eval/episode_time=0.0659]

[dg4sodfc] DataFrame:
   bxdtzg0b  cmn3go73  62raq6de  2vxp36ul
0       4.0       5.0       4.0       2.0
1      11.0       4.0       3.0      14.0
2      11.0       4.0       3.0      14.0
3       6.0       4.0       8.0       1.0
4       4.0       5.0       4.0       2.0
5       6.0       4.0       8.0       1.0
6       6.0       4.0       8.0       1.0
7       6.0       4.0       8.0       1.0
8       6.0       4.0       8.0       1.0
9       6.0       4.0       8.0       1.0
[dg4sodfc] Column Means:
bxdtzg0b    6.6
cmn3go73    4.2
62raq6de    6.2
2vxp36ul    3.8
dtype: float64
[dg4sodfc] Mean of Column Means: 5.2
[dg4sodfc] Std. of Column Means: 1.2165525060596438


In [26]:
env_key = "qc51/Breakout-v5"
all_report_dicts[env_key] = eval_via_env(env_key, eval_kwargs_override=dict(eps=eps))
display_report_results(all_report_dicts[env_key], metric_key_for_report_display)

Evaluating: sweep=sya2ty4z, run=k4hf9d66, num_eval=10


 32%|███▏      | 321/1000 [00:00<00:00, 733.21step/s, eval/step=321, eval/episode_reward=9, eval/episode_length=322, eval/episode_time=0.438]


Evaluating: sweep=sya2ty4z, run=sq95jzsj, num_eval=10


 13%|█▎        | 128/1000 [00:00<00:01, 732.57step/s, eval/step=128, eval/episode_reward=3, eval/episode_length=129, eval/episode_time=0.175]


Evaluating: sweep=sya2ty4z, run=o0fl03ok, num_eval=10


  5%|▌         | 50/1000 [00:00<00:01, 721.56step/s, eval/step=50, eval/episode_reward=1, eval/episode_length=51, eval/episode_time=0.0695]


Evaluating: sweep=sya2ty4z, run=zjjotl7m, num_eval=10


  5%|▌         | 50/1000 [00:00<00:01, 729.19step/s, eval/step=50, eval/episode_reward=1, eval/episode_length=51, eval/episode_time=0.0688]

[sya2ty4z] DataFrame:
   k4hf9d66  sq95jzsj  o0fl03ok  zjjotl7m
0       3.0       3.0       2.0       2.0
1       6.0       2.0       2.0       2.0
2       6.0       2.0       2.0       2.0
3       9.0       3.0       1.0       1.0
4       3.0       3.0       2.0       2.0
5       9.0       3.0       1.0       1.0
6       9.0       3.0       1.0       1.0
7       9.0       3.0       1.0       1.0
8       9.0       3.0       1.0       1.0
9       9.0       3.0       1.0       1.0
[sya2ty4z] Column Means:
k4hf9d66    7.2
sq95jzsj    2.8
o0fl03ok    1.4
zjjotl7m    1.4
dtype: float64
[sya2ty4z] Mean of Column Means: 3.2
[sya2ty4z] Std. of Column Means: 2.379075450674064


In [27]:
env_key = "qnrl/Breakout-v5"
all_report_dicts[env_key] = eval_via_env(env_key, eval_kwargs_override=dict(eps=eps))
display_report_results(all_report_dicts[env_key], metric_key_for_report_display)

Evaluating: sweep=eigl7wk5, run=i8ijgd1m, num_eval=10


 33%|███▎      | 326/1000 [00:07<00:16, 41.58step/s, eval/step=326, eval/episode_reward=12, eval/episode_length=327, eval/episode_time=7.82]


Evaluating: sweep=eigl7wk5, run=syrhxro1, num_eval=10


 15%|█▌        | 150/1000 [00:03<00:19, 42.67step/s, eval/step=150, eval/episode_reward=4, eval/episode_length=151, eval/episode_time=3.5]


Evaluating: sweep=eigl7wk5, run=4pc6bthv, num_eval=10


 17%|█▋        | 173/1000 [00:04<00:19, 41.48step/s, eval/step=173, eval/episode_reward=5, eval/episode_length=174, eval/episode_time=4.15]


Evaluating: sweep=eigl7wk5, run=3pv6knud, num_eval=10


 16%|█▌        | 157/1000 [00:03<00:19, 42.45step/s, eval/step=157, eval/episode_reward=4, eval/episode_length=158, eval/episode_time=3.68]


[eigl7wk5] DataFrame:
   i8ijgd1m  syrhxro1  4pc6bthv  3pv6knud
0       2.0       1.0       4.0       1.0
1       2.0       9.0       2.0       2.0
2       2.0       9.0       2.0       2.0
3      12.0       4.0       5.0       4.0
4       2.0       1.0       4.0       1.0
5      12.0       4.0       5.0       4.0
6      12.0       4.0       5.0       4.0
7      12.0       4.0       5.0       4.0
8      12.0       4.0       5.0       4.0
9      12.0       4.0       5.0       4.0
[eigl7wk5] Column Means:
i8ijgd1m    8.0
syrhxro1    4.4
4pc6bthv    4.2
3pv6knud    3.0
dtype: float64
[eigl7wk5] Mean of Column Means: 4.9
[eigl7wk5] Std. of Column Means: 1.8681541692269403
